In [ ]:
# Imports 
import os, math, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

# Import models from scripts
from scripts.Recommendation_System.models import BasicCollaborativeFiltering, AdditiveHybridMFWithText, MLPModel


In [ ]:
# Load processed data (assumes prior processing)
review_path = '../data/processed/review_data.jsonl'
meta_path = '../data/processed/metadata.jsonl'
review_df = pd.read_json(review_path, lines=True)
metadata_df = pd.read_json(meta_path, lines=True)

# Build mappings
user_ids = review_df['user_id'].unique()
item_ids = review_df['parent_asin'].unique()
user2idx = {u: i for i, u in enumerate(user_ids)}
item2idx = {a: i for i, a in enumerate(item_ids)}
idx2item = {i: a for a, i in item2idx.items()}

# Encode indices
review_df['user_idx'] = review_df['user_id'].map(user2idx)
review_df['item_idx'] = review_df['parent_asin'].map(item2idx)

# Metadata prep
metadata_df = metadata_df[['parent_asin', 'main_category', 'average_rating', 'rating_number', 'price', 'title']].copy()
metadata_df['price'] = pd.to_numeric(metadata_df['price'], errors='coerce')
metadata_df['average_rating'] = pd.to_numeric(metadata_df['average_rating'], errors='coerce').fillna(0)
metadata_df['rating_number'] = pd.to_numeric(metadata_df['rating_number'], errors='coerce').fillna(0).astype(int)

from sklearn.preprocessing import StandardScaler
metadata_df['price_log'] = np.log1p(metadata_df['price'])
metadata_df['rating_number_log'] = np.log1p(metadata_df['rating_number'])
for col in ['price_log', 'average_rating', 'rating_number_log']:
    scaler = StandardScaler()
    metadata_df[col + '_scaled'] = scaler.fit_transform(metadata_df[[col]])

# Category ids
main_categories = metadata_df['main_category'].fillna('Unknown').astype(str)
cat2idx = {cat: idx+1 for idx, cat in enumerate(main_categories.unique())}
cat2idx['<unk>'] = 0
metadata_df['main_cat_idx'] = main_categories.map(lambda c: cat2idx.get(c, 0))

# Merge
merged_df = review_df.merge(
    metadata_df[['parent_asin', 'main_cat_idx', 'price_log_scaled', 'average_rating_scaled', 'rating_number_log_scaled']],
    on='parent_asin', how='left'
)

# Leave-one-out split (validation)
merged_df = merged_df.sort_values(by=['user_id', 'reviewTime'])
val_df = merged_df.groupby('user_id').tail(1)
train_df = merged_df.drop(val_df.index)


In [ ]:
# Dimensions and constants
n_users = int(max(train_df['user_idx'].max(), val_df['user_idx'].max())) + 1
n_items = int(max(train_df['item_idx'].max(), val_df['item_idx'].max())) + 1
n_main_cats = int(max(train_df['main_cat_idx'].max(), val_df['main_cat_idx'].max())) + 1

BATCH_SIZE = 1024
N_NEGATIVES = 5
sbert_dim = 384

# Item-level arrays for features
item_cat = np.zeros(n_items, dtype=np.int64)
item_price = np.zeros(n_items, dtype=np.float32)
item_avg = np.zeros(n_items, dtype=np.float32)
item_count = np.zeros(n_items, dtype=np.float32)
meta_src = train_df[['item_idx','main_cat_idx','price_log_scaled','average_rating_scaled','rating_number_log_scaled']].drop_duplicates('item_idx')
for _, row in meta_src.iterrows():
    i = int(row.item_idx)
    if i < n_items:
        item_cat[i] = int(row.main_cat_idx) if pd.notna(row.main_cat_idx) else 0
        item_price[i] = float(row.price_log_scaled) if pd.notna(row.price_log_scaled) else 0.0
        item_avg[i] = float(row.average_rating_scaled) if pd.notna(row.average_rating_scaled) else 0.0
        item_count[i] = float(row.rating_number_log_scaled) if pd.notna(row.rating_number_log_scaled) else 0.0
item_price = np.nan_to_num(item_price, nan=0.0)
item_avg = np.nan_to_num(item_avg, nan=0.0)
item_count = np.nan_to_num(item_count, nan=0.0)

item_cat_t = torch.from_numpy(item_cat)
item_price_t = torch.from_numpy(item_price)
item_avg_t = torch.from_numpy(item_avg)
item_count_t = torch.from_numpy(item_count)

# Training tensors
u_arr = torch.from_numpy(train_df['user_idx'].values.astype(np.int64))
pos_item_arr = torch.from_numpy(train_df['item_idx'].values.astype(np.int64))
pos_cat_arr = torch.from_numpy(train_df['main_cat_idx'].values.astype(np.int64))
pos_price_arr = torch.nan_to_num(torch.from_numpy(train_df['price_log_scaled'].values.astype(np.float32)), nan=0.0)
pos_avg_arr = torch.nan_to_num(torch.from_numpy(train_df['average_rating_scaled'].values.astype(np.float32)), nan=0.0)
pos_count_arr = torch.nan_to_num(torch.from_numpy(train_df['rating_number_log_scaled'].values.astype(np.float32)), nan=0.0)

# User -> positives
user_pos = {}
for u, i in zip(u_arr.numpy(), pos_item_arr.numpy()):
    user_pos.setdefault(int(u), set()).add(int(i))


In [ ]:
# Title embeddings (CPU-only)
from sentence_transformers import SentenceTransformer

emb_path = '../data/embeddings/title_embeddings.npy'
if os.path.exists(emb_path):
    title_embeddings = np.load(emb_path)
    title_embeddings_tensor = torch.from_numpy(title_embeddings).float()
else:
    sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
    sbert_dim = sbert_model.get_sentence_embedding_dimension()
    os.makedirs('../data/embeddings', exist_ok=True)
    titles_df = metadata_df[['parent_asin', 'title']].copy()
    titles_df['title'] = titles_df['title'].fillna('Unknown Product').astype(str)
    title_lookup = dict(zip(titles_df['parent_asin'], titles_df['title']))
    item_titles = ['Unknown Product'] * n_items
    for idx in range(n_items):
        asin = idx2item.get(idx, None)
        if asin is not None:
            item_titles[idx] = title_lookup.get(asin, 'Unknown Product')
    title_embeddings = sbert_model.encode(item_titles, show_progress_bar=True, convert_to_numpy=True)
    np.save(emb_path, title_embeddings)
    title_embeddings_tensor = torch.from_numpy(title_embeddings).float()


In [ ]:
# Utilities: loss, negatives, validation eval
softplus = nn.Softplus()

def bpr_loss(pos_scores, neg_scores):
    return softplus(neg_scores - pos_scores).mean()

def sample_multiple_negatives(user_batch, n_items, user_pos_sets, n_negatives=5):
    B = user_batch.size(0)
    neg_items = torch.randint(0, n_items, (B, n_negatives))
    for idx, u in enumerate(user_batch.tolist()):
        positives = user_pos_sets.get(u, set())
        for k in range(n_negatives):
            tries = 0
            while neg_items[idx, k].item() in positives and tries < 10:
                neg_items[idx, k] = torch.randint(0, n_items, (1,))
                tries += 1
    return neg_items

def evaluate_validation_sampled(model, val_df, model_type, K=10, n_negatives=100):
    if len(val_df) == 0:
        return 0.0, 0.0
    model.eval()
    hits, ndcgs = [], []
    with torch.no_grad():
        for _, row in val_df.iterrows():
            u = int(row['user_idx']); true_item = int(row['item_idx'])
            neg_items = np.random.choice(n_items, size=n_negatives, replace=True)
            candidate_items = [true_item] + [ni for ni in neg_items if ni != true_item]
            candidate_tensor = torch.tensor(candidate_items, dtype=torch.long)
            user_tensor = torch.full((len(candidate_items),), u, dtype=torch.long)
            if model_type == 'CF':
                scores = model.score(user_tensor, candidate_tensor)
            elif model_type in ['Text','MLP']:
                scores = model.score(user_tensor, candidate_tensor,
                                     item_cat_t[candidate_tensor], item_price_t[candidate_tensor],
                                     item_avg_t[candidate_tensor], item_count_t[candidate_tensor],
                                     title_embeddings_tensor[candidate_tensor])
            else:
                scores = model.score(user_tensor, candidate_tensor,
                                     item_cat_t[candidate_tensor], item_price_t[candidate_tensor],
                                     item_avg_t[candidate_tensor], item_count_t[candidate_tensor])
            top_k = min(K, len(candidate_items))
            _, top_indices = torch.topk(scores, k=top_k)
            top_items = [candidate_items[idx.item()] for idx in top_indices]
            hit = 1.0 if true_item in top_items else 0.0
            ndcg = 1.0 / math.log2(top_items.index(true_item) + 2) if true_item in top_items else 0.0
            hits.append(hit); ndcgs.append(ndcg)
    return (float(np.mean(hits)) if hits else 0.0, float(np.mean(ndcgs)) if ndcgs else 0.0)

In [ ]:
# Training loop (single-seed per config with early stopping)

def train_and_validate(model_class, model_params, model_type, lr, l2, max_epochs=50, patience=3):
    model = model_class(**model_params)
    model.train()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=l2)

    indices = np.arange(len(u_arr))
    best_state, best_ndcg = None, -1.0
    patience_counter = 0

    for epoch in range(1, max_epochs + 1):
        np.random.shuffle(indices)
        for start in range(0, len(indices), BATCH_SIZE):
            batch_idx = indices[start:start+BATCH_SIZE]
            if len(batch_idx) == 0:
                continue
            ub = u_arr[batch_idx]
            pb = pos_item_arr[batch_idx]
            neg_b = sample_multiple_negatives(ub, n_items, user_pos, N_NEGATIVES)
            optimizer.zero_grad()
            losses = []
            for k in range(N_NEGATIVES):
                nb = neg_b[:, k]
                if model_type == 'CF':
                    pos_scores = model.score(ub, pb)
                    neg_scores = model.score(ub, nb)
                elif model_type in ['Text','MLP']:
                    pos_scores = model.score(ub, pb, pos_cat_arr[batch_idx], pos_price_arr[batch_idx], pos_avg_arr[batch_idx], pos_count_arr[batch_idx], title_embeddings_tensor[pb])
                    neg_scores = model.score(ub, nb, item_cat_t[nb], item_price_t[nb], item_avg_t[nb], item_count_t[nb], title_embeddings_tensor[nb])
                else:
                    pos_scores = model.score(ub, pb, pos_cat_arr[batch_idx], pos_price_arr[batch_idx], pos_avg_arr[batch_idx], pos_count_arr[batch_idx])
                    neg_scores = model.score(ub, nb, item_cat_t[nb], item_price_t[nb], item_avg_t[nb], item_count_t[nb])
                losses.append(bpr_loss(pos_scores, neg_scores))
            loss = torch.stack(losses).mean()
            loss.backward()
            optimizer.step()

        model.eval()
        hr, ndcg = evaluate_validation_sampled(model, val_df, model_type)
        if ndcg > best_ndcg:
            best_ndcg = ndcg
            patience_counter = 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            patience_counter += 1
        if patience_counter >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, {'val_hr': hr, 'val_ndcg': best_ndcg}

# Define search space (concise)
search_space = {
    'CF': [
        {'emb_dim': 64, 'dropout': 0.1, 'lr': 5e-4, 'l2': 1e-6},
        {'emb_dim': 128, 'dropout': 0.1, 'lr': 5e-4, 'l2': 1e-6}
    ],
    'Baseline': [
        {'emb_dim': 64, 'dropout': 0.1, 'lr': 5e-4, 'l2': 1e-6},
        {'emb_dim': 32, 'dropout': 0.2, 'lr': 1e-3, 'l2': 1e-6}
    ],
    'Text': [
        {'emb_dim': 32, 'text_proj_dim': 32, 'dropout': 0.2, 'lr': 1e-3, 'l2': 1e-6},
        {'emb_dim': 64, 'text_proj_dim': 64, 'dropout': 0.1, 'lr': 5e-4, 'l2': 1e-6}
    ],
    'MLP': [
        {'emb_dim': 64, 'text_proj_dim': 64, 'mlp_dim': 64, 'hidden_dim': 128, 'dropout': 0.1, 'lr': 5e-4, 'l2': 1e-6},
        {'emb_dim': 32, 'text_proj_dim': 32, 'mlp_dim': 64, 'hidden_dim': 128, 'dropout': 0.2, 'lr': 1e-3, 'l2': 1e-6}
    ]
}

results = []

for model_name, configs in search_space.items():
    for cfg in configs:
        if model_name == 'CF':
            params = {'n_users': n_users, 'n_items': n_items, 'emb_dim': cfg['emb_dim'], 'dropout': cfg['dropout']}
            cls = BasicCollaborativeFiltering
        elif model_name == 'MLP':
            params = {'n_users': n_users, 'n_items': n_items, 'n_main_cats': n_main_cats,
                      'text_emb_dim': sbert_dim, 'emb_dim': cfg['emb_dim'], 'text_proj_dim': cfg['text_proj_dim'],
                      'mlp_dim': cfg['mlp_dim'], 'hidden_dim': cfg['hidden_dim'],
                      'use_avg_rating': True, 'use_rating_count': True, 'use_price': True,
                      'use_text': True, 'dropout': cfg['dropout']}
            cls = MLPModel
        elif model_name == 'Text':
            params = {'n_users': n_users, 'n_items': n_items, 'n_main_cats': n_main_cats,
                      'text_emb_dim': sbert_dim, 'emb_dim': cfg['emb_dim'], 'text_proj_dim': cfg['text_proj_dim'],
                      'use_avg_rating': True, 'use_rating_count': True, 'use_price': True,
                      'use_text': True, 'dropout': cfg['dropout']}
            cls = AdditiveHybridMFWithText
        else:  # Baseline
            params = {'n_users': n_users, 'n_items': n_items, 'n_main_cats': n_main_cats,
                      'text_emb_dim': sbert_dim, 'emb_dim': cfg['emb_dim'],
                      'use_avg_rating': True, 'use_rating_count': True, 'use_price': True,
                      'use_text': False, 'dropout': cfg['dropout']}
            cls = AdditiveHybridMFWithText

        model, metrics = train_and_validate(cls, params, model_name, cfg['lr'], cfg['l2'], max_epochs=50, patience=3)
        results.append({
            'model': model_name,
            **cfg,
            **metrics
        })

res_df = pd.DataFrame(results).sort_values(by=['val_ndcg'], ascending=False)

# Save concise results
csv_path = './advanced_search_results_clean.csv'
res_df.to_csv(csv_path, index=False)
